# Your own model: one Qwen for everything

Fine-tunes **Qwen3-VL-4B-Instruct** (the Qwen that can also see) into one model that knows everything in
`qwen_training.zip`: the Pokémon TCG and its rulings, the Pokémon games, Thomas & Friends, Unity, C#, Python, Git, GBA/DS
games, and, if the pack has pictures, how to rebuild a picture as a Unity scene. It saves two `.gguf` files (the model
and its image part) that `install_to_ollama.py` makes into one Ollama model.

**On Google Colab (free):**
1. On your PC: `python build_dataset.py` writes `qwen_training.zip` (everything in one file). Put it at the top of your
   Google Drive (*My Drive*): it's found there automatically, which is the reliable way for a big file. (Or leave it
   off Drive and upload it when the notebook asks.)
2. *Runtime → Change runtime type → T4 GPU*, then *Save*.
3. *Runtime → Run all*. With everything, training takes roughly 4–6 hours. It saves a checkpoint to your Drive every
   250 steps: if Colab disconnects, just *Run all* again and it carries on where it stopped.
4. At the end the two `.gguf` files (about 2.5 GB and 0.8 GB) are copied to your Drive.

Try `MAX_STEPS = 60` first for a quick run (a few minutes) to check that everything works.

**On your own PC:** run it with Jupyter under WSL2 (Unsloth needs Linux), with `qwen_training.zip` next to it.

In [ ]:
%%capture
# Unsloth: fast LoRA training that fits small GPUs. (Already installed? This just checks.)
!pip install unsloth

In [ ]:
# ---- Settings --------------------------------------------------------------------------------------------------
BASE_MODEL = "unsloth/Qwen3-VL-4B-Instruct"   # the same model as Ollama's qwen3-vl:4b-instruct
MAX_SEQ_LENGTH = 2048             # longest example in tokens (the pack's are well under this)
EPOCHS = 1                        # passes over the data: 1-2; more risks forgetting general skills
MAX_STEPS = 0                     # 0 = train the full EPOCHS; e.g. 60 for a quick test run first
GENERAL_SHARE = 0.15              # share of general conversations mixed in, so it stays good at normal chat
PICTURE_REPEAT = 1                # how many times each picture is used: 2 or 3 if the pack has few (under 2,000)
CHECK_PICTURES = 5                # held-back pictures scored before and after training (about 30 s each)
OUTPUT_NAME = "pokemon-tcg"
SAVE_TO_DRIVE = True              # checkpoints and the .gguf files go to Google Drive (survives disconnects)
CHECKPOINT_EVERY = 250            # steps between checkpoints (when SAVE_TO_DRIVE is on)
PACK = "qwen_training.zip"

In [ ]:
import glob, json, os, random, sys, zipfile
OUTPUT_DIR = "outputs"
DRIVE = "/content/drive/MyDrive"
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTPUT_DIR = f"{DRIVE}/llm-checkpoints/{OUTPUT_NAME}"
    os.makedirs(OUTPUT_DIR, exist_ok=True)
found = [p for p in (PACK, f"{DRIVE}/{PACK}") if os.path.exists(p)]
if not found:
    from google.colab import files
    print(f"Upload {PACK} (from PokemonTCGSim/LlmTraining):")
    files.upload()
    found = [PACK]
with zipfile.ZipFile(found[0]) as z:
    z.extractall("pack")
rows = [json.loads(line) for line in open("pack/train.jsonl", encoding="utf-8")]
checks = [json.loads(line) for line in open("pack/pictures/check.jsonl", encoding="utf-8")] if os.path.exists("pack/pictures/check.jsonl") else []
names = json.load(open("pack/pictures/sprite_names.json")) if checks else []
n_pictures = sum(1 for r in rows if "image" in r)
sys.path.insert(0, "pack/tools")
print(f"{found[0]}: {len(rows):,} examples, {n_pictures:,} of them pictures ({len(checks)} more held back for checking)")

In [ ]:
from unsloth import FastVisionModel
import torch

model, tokenizer = FastVisionModel.from_pretrained(
    BASE_MODEL, max_seq_length=MAX_SEQ_LENGTH, load_in_4bit=True, use_gradient_checkpointing="unsloth")
# LoRA: the base model stays frozen; only this small add-on learns (which is what keeps it from forgetting much).
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers=n_pictures > 0,   # the image part learns your sprites (only when there are pictures)
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    r=16, lora_alpha=16, lora_dropout=0, bias="none", random_state=3407)

In [ ]:
from datasets import Dataset, load_dataset
from PIL import Image

def parts(text):
    return [{"type": "text", "text": text}] if isinstance(text, str) else text

def conversation(row):
    # A conversation in the vision model's format; a picture goes in front of the question, as ask_model.py sends it.
    msgs = [{"role": m["role"], "content": parts(m["content"])} for m in row["messages"]]
    if "image" in row:
        user = next(m for m in msgs if m["role"] == "user")
        user["content"].insert(0, {"type": "image", "image": Image.open(os.path.join("pack", row["image"])).convert("RGB")})
    return msgs

# General conversations (a public instruction dataset), so it keeps its everyday skills.
general_count = int(len(rows) * GENERAL_SHARE / (1 - GENERAL_SHARE))
general = []
if general_count > 0:
    roles = {"human": "user", "gpt": "assistant", "system": "system", "user": "user", "assistant": "assistant"}
    g = load_dataset("mlabonne/FineTome-100k", split="train").shuffle(seed=3407)
    for convo in g.select(range(min(general_count, len(g))))["conversations"]:
        general.append({"messages": [{"role": roles[m["from"]], "content": m["value"]} for m in convo]})

items = rows + general + [r for r in rows if "image" in r] * (PICTURE_REPEAT - 1)
random.Random(3407).shuffle(items)

# The trainer needs a Hugging Face Dataset. It holds the text and each picture's path; a picture is opened only when its
# example is used (with_transform), so thousands of pictures don't fill the memory.
table = Dataset.from_list([{"messages": r["messages"], "image": r.get("image", "")} for r in items])
dataset = table.with_transform(lambda batch: {"messages": [
    conversation({"messages": m, "image": i} if i else {"messages": m}) for m, i in zip(batch["messages"], batch["image"])]})
print(f"{len(rows):,} examples from the pack + {len(general):,} general = {len(dataset):,}")

In [ ]:
# Scoring pictures: the model rebuilds a few held-back pictures, scored against their real scenes (see VisionTools/score.py).
def ask_picture(row):
    msgs = conversation(row)[:2]   # the system prompt and the question with its picture
    image = msgs[1]["content"][0]["image"]
    msgs[1]["content"][0] = {"type": "image"}
    prompt = tokenizer.apply_chat_template(msgs, add_generation_prompt=True)
    inputs = tokenizer(image, prompt, add_special_tokens=False, return_tensors="pt").to("cuda")
    out = model.generate(**inputs, max_new_tokens=1500, do_sample=False, use_cache=True)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

def check_pictures(label):
    if not checks or CHECK_PICTURES <= 0:
        return
    import scenespec, score
    FastVisionModel.for_inference(model)
    results = []
    for row in checks[:CHECK_PICTURES]:
        answer = ask_picture(row)
        try:
            guess = scenespec.parse_answer(answer)
            if scenespec.problems(guess, names):
                guess = None
        except ValueError:
            guess = None
        results.append(score.score(json.loads(row["messages"][-1]["content"]), guess))
    print(label + ": " + score.describe(score.summary(results)))

check_pictures("Pictures before training")

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.trainer import UnslothVisionDataCollator

FastVisionModel.for_training(model)
trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    data_collator=UnslothVisionDataCollator(
        model, tokenizer,
        resize="max",                          # keep the pictures' size (make_dataset keeps them at 640 or less)
        train_on_responses_only=True,          # learn only from the answers, not from repeating the questions
        instruction_part="<|im_start|>user\n",
        response_part="<|im_start|>assistant\n",
    ),
    train_dataset=dataset,
    args=SFTConfig(
        per_device_train_batch_size=2, gradient_accumulation_steps=4,
        warmup_steps=10, num_train_epochs=EPOCHS, max_steps=MAX_STEPS if MAX_STEPS > 0 else -1,
        learning_rate=2e-4, logging_steps=20, optim="adamw_8bit", weight_decay=0.01,
        lr_scheduler_type="linear", seed=3407, output_dir=OUTPUT_DIR, report_to="none",
        bf16=torch.cuda.is_bf16_supported(),   # the free T4 has no bf16: fp16 there
        fp16=not torch.cuda.is_bf16_supported(),
        save_strategy="steps" if SAVE_TO_DRIVE else "no", save_steps=CHECKPOINT_EVERY, save_total_limit=2,
        remove_unused_columns=False, dataset_text_field="", dataset_kwargs={"skip_prepare_dataset": True},
        max_length=MAX_SEQ_LENGTH))
# Carry on from the last checkpoint if an earlier run was cut off (same pack, same settings).
resume = SAVE_TO_DRIVE and bool(glob.glob(os.path.join(OUTPUT_DIR, "checkpoint-*")))
if resume:
    print("Resuming from the last checkpoint in", OUTPUT_DIR)
stats = trainer.train(resume_from_checkpoint=resume or None)
print(stats)

In [ ]:
# Try it: a few questions, and the held-back pictures again.
FastVisionModel.for_inference(model)
system = rows[0]["messages"][0]["content"]
for q in ["Can Rare Candy be used on the first turn of the game?",
          "If my Active Pokémon is Asleep, can I retreat it?",
          "What type is Gengar and what is it weak to?",
          "Who is Edward in Thomas & Friends?",
          "How do I use Transform.Rotate in Unity?",
          "How do I undo my last Git commit but keep the changes?",
          "What is the capital of France?"]:
    msgs = [{"role": "system", "content": parts(system)}, {"role": "user", "content": parts(q)}]
    prompt = tokenizer.apply_chat_template(msgs, add_generation_prompt=True)
    inputs = tokenizer(text=prompt, add_special_tokens=False, return_tensors="pt").to("cuda")
    out = model.generate(**inputs, max_new_tokens=200, do_sample=True, temperature=0.7, top_p=0.8)
    print("Q:", q)
    print("A:", tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip(), "\n")
check_pictures("Pictures after training")

In [ ]:
# Save: the trained add-on merged into the model, as two 4-bit .gguf files for Ollama (takes several minutes):
# the model itself, and its image part ("mmproj"). install_to_ollama.py puts them together.
import shutil
model.save_pretrained(f"{OUTPUT_DIR}/lora")      # the small add-on too, to train it further later
tokenizer.save_pretrained(f"{OUTPUT_DIR}/lora")
model.save_pretrained_gguf(OUTPUT_NAME, tokenizer, quantization_method="q4_k_m")
ggufs = sorted(set(glob.glob(f"{OUTPUT_NAME}*/**/*.gguf", recursive=True) + glob.glob("*.gguf")))
projector = next(p for p in ggufs if "mmproj" in os.path.basename(p).lower())
main_file = next(p for p in ggufs if "mmproj" not in os.path.basename(p).lower() and "q4_k_m" in p.lower())
finals = {main_file: f"{OUTPUT_NAME}.Q4_K_M.gguf", projector: f"{OUTPUT_NAME}.mmproj.gguf"}
for src, dst in finals.items():
    if os.path.abspath(src) != os.path.abspath(dst):
        shutil.copy(src, dst)
    print("saved", dst, os.path.getsize(dst) // (1024 * 1024), "MB")

In [ ]:
try:
    if SAVE_TO_DRIVE:
        for f in finals.values():
            shutil.copy(f, f"{DRIVE}/{f}")
            print("Copied to Google Drive: MyDrive/" + f)
    else:
        from google.colab import files
        for f in finals.values():
            files.download(f)
except ImportError:
    print("Not on Colab: the files are here:", os.path.abspath("."))
print(f"Then on your PC, with both files in one folder: python install_to_ollama.py {OUTPUT_NAME}.Q4_K_M.gguf")